In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd

df = pd.read_csv('kidney_disease_cleaned.csv')
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,0,0,0,0,121.000000,...,44.0,7800.0,5.200000,1,1,0,1,0,0,0
1,7.0,50.0,1.020,4.0,0.0,0,0,0,0,147.864407,...,38.0,6000.0,4.702247,0,0,0,1,0,0,0
2,62.0,80.0,1.010,2.0,3.0,0,0,0,0,423.000000,...,31.0,7500.0,4.702247,0,1,0,0,0,1,0
3,48.0,70.0,1.005,4.0,0.0,0,1,1,0,117.000000,...,32.0,6700.0,3.900000,1,0,0,0,1,1,0
4,51.0,80.0,1.010,2.0,0.0,0,0,0,0,106.000000,...,35.0,7300.0,4.600000,0,0,0,1,0,0,0


In [5]:
x = df.drop('class',axis=1)
y = df['class']

In [6]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

#### Create a copy for Tree-Based Models (No Scaling)


In [7]:
x_train_tree = x_train.copy()
x_test_tree = x_test.copy()

#### Create a copy for Scale-Based Models

In [8]:
x_train_scale = x_train.copy()
x_test_scale = x_test.copy()

In [9]:
x.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane'],
      dtype='object')

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

continous_cols = ['age','bp','sg','al','su','bgr','bu','sc','sod','pot','hemo','pcv','wbcc','rbcc']
x_train_scale[continous_cols] = scaler.fit_transform(x_train_scale[continous_cols])
x_test_scale[continous_cols] = scaler.transform(x_test_scale[continous_cols])

## Logistic Regression

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [ ]:
lr = LogisticRegression(random_state=42)
param_grid = [
    {
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'class_weight': [None, 'balanced'],
        'max_iter': [500, 1000]
    },
    {
        'solver': ['lbfgs'],
        'penalty': ['l2', None],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'class_weight': [None, 'balanced'],
        'max_iter': [500, 1000]
    },
    {
        'solver': ['saga'],
        'penalty': ['l1', 'l2', 'elasticnet'],
        'l1_ratio': [0.2, 0.5, 0.8],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'class_weight': [None, 'balanced'],
        'max_iter': [500, 1000]
    }
]

grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

grid_lr.fit(x_train_scale, y_train)

# print("Best Parameters:")
# print(grid_lr.best_params_)

# print("\nBest Cross Validation Score:")
# print(grid_lr.best_score_)

Fitting 5 folds for each of 312 candidates, totalling 1560 fits


In [ ]:
best_lr = grid_lr.best_estimator_
y_pred = best_lr.predict(x_test_scale)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9875

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        50
           1       0.97      1.00      0.98        30

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80


Confusion Matrix:
[[49  1]
 [ 0 30]]


In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = best_lr.predict_proba(x_test_scale)[:,1]
roc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", roc)

ROC-AUC Score: 1.0


## KNeighborsClassifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
knn = KNeighborsClassifier()
param_grid = [
    {
        'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    {
        'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['minkowski'],
        'p': [1, 2]
    }
]

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

grid_knn.fit(x_train_scale, y_train)

print("Best Parameters:")
print(grid_knn.best_params_)

print("\nBest Cross Validation Score:")
print(grid_knn.best_score_)

Fitting 5 folds for each of 56 candidates, totalling 280 fits
Best Parameters:
{'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}

Best Cross Validation Score:
0.9715918367346938


In [ ]:
best_knn = grid_knn.best_estimator_
y_pred = best_knn.predict(x_test_scale)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

y_prob = best_knn.predict_proba(x_test_scale)[:, 1]
roc = roc_auc_score(y_test, y_prob)

print("\nROC-AUC Score:", roc)

Accuracy: 0.9625

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        50
           1       0.91      1.00      0.95        30

    accuracy                           0.96        80
   macro avg       0.95      0.97      0.96        80
weighted avg       0.97      0.96      0.96        80


Confusion Matrix:
[[47  3]
 [ 0 30]]

ROC-AUC Score: 0.988


## SVM

In [ ]:

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
svc = SVC(probability=True, random_state=42)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.1]
}

grid_svc = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)
grid_svc.fit(x_train_scale, y_train)

print("Best Parameters:")
print(grid_svc.best_params_)

print("\nBest Cross Validation Score:")
print(grid_svc.best_score_)


Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best Parameters:
{'C': 0.1, 'gamma': 0.1, 'kernel': 'rbf'}

Best Cross Validation Score:
0.9936507936507937


In [ ]:
best_svc = grid_svc.best_estimator_
y_pred = best_svc.predict(x_test_scale)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

y_prob = best_svc.predict_proba(x_test_scale)[:, 1]
roc = roc_auc_score(y_test, y_prob)

print("\nROC-AUC Score:", roc)

Accuracy: 0.9875

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        50
           1       0.97      1.00      0.98        30

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80


Confusion Matrix:
[[49  1]
 [ 0 30]]

ROC-AUC Score: 1.0


## DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV


dt = DecisionTreeClassifier(random_state=42)


param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt']
}


grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_dt.fit(x_train_tree, y_train)
print("Best Parameters:")
print(grid_dt.best_params_)

print("\nBest Cross Validation Score:")
print(grid_dt.best_score_)

Fitting 5 folds for each of 180 candidates, totalling 900 fits
Best Parameters:
{'criterion': 'entropy', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}

Best Cross Validation Score:
0.9778769841269842


In [ ]:
best_dt = grid_dt.best_estimator_

y_pred = best_dt.predict(x_test_tree)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

y_prob = best_dt.predict_proba(x_test_tree)[:, 1]
roc = roc_auc_score(y_test, y_prob)

print("\nROC-AUC Score:", roc)

Accuracy: 0.95

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96        50
           1       0.93      0.93      0.93        30

    accuracy                           0.95        80
   macro avg       0.95      0.95      0.95        80
weighted avg       0.95      0.95      0.95        80


Confusion Matrix:
[[48  2]
 [ 2 28]]

ROC-AUC Score: 0.9466666666666667


## RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)


param_grid = {
    'n_estimators': [100, 200],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)
grid_rf.fit(x_train_tree, y_train)

print("Best Parameters:")
print(grid_rf.best_params_)

print("\nBest Cross Validation Score:")
print(grid_rf.best_score_)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best Parameters:
{'bootstrap': True, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

Best Cross Validation Score:
0.9916630481980026


In [ ]:
best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(x_test_tree)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

y_prob = best_rf.predict_proba(x_test_tree)[:, 1]
roc = roc_auc_score(y_test, y_prob)

print("\nROC-AUC Score:", roc)

Accuracy: 0.975

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        50
           1       1.00      0.93      0.97        30

    accuracy                           0.97        80
   macro avg       0.98      0.97      0.97        80
weighted avg       0.98      0.97      0.97        80


Confusion Matrix:
[[50  0]
 [ 2 28]]

ROC-AUC Score: 0.9993333333333333


## ExtraTreesClassifier

In [ ]:
# from sklearn.ensemble import ExtraTreesClassifier
# from sklearn.model_selection import GridSearchCV

# et = ExtraTreesClassifier(random_state=42)

# param_grid = {
#     'n_estimators': [100, 200],
#     'criterion': ['gini', 'entropy'],
#     'max_depth': [None, 5, 10],
#     'min_samples_split': [2, 5],
#     'min_samples_leaf': [1, 2],
#     'max_features': ['sqrt', 'log2']
# }

# grid_et = GridSearchCV(
#     estimator=et,
#     param_grid=param_grid,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1,
#     verbose=2
# )

# from sklearn.ensemble import ExtraTreesClassifier
# from sklearn.model_selection import GridSearchCV


# et = ExtraTreesClassifier(random_state=42)


# param_grid = {
#     'n_estimators': [100, 200],
#     'criterion': ['gini', 'entropy'],
#     'max_depth': [None, 5, 10],
#     'min_samples_split': [2, 5],
#     'min_samples_leaf': [1, 2],
#     'max_features': ['sqrt', 'log2']
# }

# grid_et = GridSearchCV(
#     estimator=et,
#     param_grid=param_grid,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1,
#     verbose=2
# )

# grid_et.fit(x_train_tree, y_train)

# print("Best Parameters:")
# print(grid_et.best_params_)

# print("\nBest Cross Validation Score:")
# print(grid_et.best_score_)


Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best Parameters:
{'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Best Cross Validation Score:
1.0


In [ ]:
# best_et = grid_et.best_estimator_

# # y_pred = best_et.predict(x_test_tree)

In [ ]:
# print("Accuracy:", accuracy_score(y_test, y_pred))

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

# y_prob = best_et.predict_proba(x_test_tree)[:, 1]
# roc = roc_auc_score(y_test, y_prob)

# print("\nROC-AUC Score:", roc)

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        30

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80


Confusion Matrix:
[[50  0]
 [ 0 30]]

ROC-AUC Score: 1.0


In [ ]:
# from sklearn.model_selection import cross_val_score
# from sklearn.ensemble import ExtraTreesClassifier

# model = ExtraTreesClassifier(random_state=42)

# scores = cross_val_score(model, x, y, cv=5, scoring='accuracy')

# print("Fold Accuracies:", scores)
# print("Mean Accuracy:", scores.mean())


Fold Accuracies: [1.         1.         1.         0.98734177 1.        ]
Mean Accuracy: 0.9974683544303797
